# SerumGate-Minus Self-Pair Same-Context Evaluation

Evaluate trained H1N1 SerumGate-Minus checkpoints on strict self-pair sanity-test rows. The expected antigenic distance is `0` because each row uses the same serum/reference virus as the query virus in the same passage/context.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

REPO = Path("/home/chenyh/workspace/fluProfiler")
sys.path.insert(0, str(REPO / "experiments" / "serum_gate"))

from train_zero_shot_minus import (
    SerumGateVocabs,
    SerumGateMinusConfig,
    SerumGateMinusModel,
    build_loader,
    evaluate_model,
    load_embeddings,
)

TEST_CSV = REPO / "data/dataset/H1H3_HA1/splited/20260707_170858/serum_self_pair_same_context/test.csv"
EMBEDDING_DIR = REPO / "data/embedding/files"

MODELS = {
    "Minus128": REPO / "results/H1H3_HA1/SerumGate-Minus/serum/subtype/H1N1/checkpoints/best_model.pth",
    "Minus16": REPO / "results/H1H3_HA1/SerumGate-Minus-latent16/serum/subtype/H1N1/checkpoints/best_model.pth",
    "Minus8": REPO / "results/H1H3_HA1/SerumGate-Minus-latent8/serum/subtype/H1N1/checkpoints/best_model.pth",
}

OUT_DIR = REPO / "results/H1H3_HA1/self_pair_same_context_H1N1_inference"
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device


device(type='cuda', index=0)

In [2]:
test_df = pd.read_csv(TEST_CSV, low_memory=False)
test_df = test_df[test_df["Type"].astype(str).eq("H1N1")].reset_index(drop=True)

checks = {
    "rows": len(test_df),
    "label_values": sorted(test_df["label"].unique().tolist()),
    "seq_id_self": bool((test_df["seq_id_a"].astype(str) == test_df["seq_id_c"].astype(str)).all()),
    "HA_self": bool((test_df["serumHA"].astype(str) == test_df["virusHA"].astype(str)).all()),
    "passage_self": bool((test_df["serumPassCat"].astype(str) == test_df["virusPassCat"].astype(str)).all()),
    "name_self": bool((test_df["serumName"].astype(str) == test_df["virusName"].astype(str)).all()),
}
checks


{'rows': 500,
 'label_values': [0.0],
 'seq_id_self': True,
 'HA_self': True,
 'passage_self': True,
 'name_self': True}

In [3]:
assert checks["rows"] > 0
assert checks["label_values"] == [0.0]
assert checks["seq_id_self"]
assert checks["HA_self"]
assert checks["passage_self"]

seq_ids = pd.concat([test_df["seq_id_a"], test_df["seq_id_c"]]).dropna().astype(str).unique()
embedding_files = [f"matrix_{seq_id}.pt" for seq_id in sorted(seq_ids)]

embeddings = load_embeddings(EMBEDDING_DIR, embedding_files, show_progress=True)
len(embeddings)


loading embeddings: 100%|██████████| 13/13 [00:00<00:00, 40.33it/s]


13

In [4]:
def run_checkpoint(model_name: str, ckpt_path: Path) -> tuple[dict, pd.DataFrame]:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    config = SerumGateMinusConfig(**ckpt["model_config"])
    model = SerumGateMinusModel(config)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    vocabs = SerumGateVocabs(
        passage_to_id=ckpt["passage_to_id"],
        subtype_to_id=ckpt["subtype_to_id"],
    )

    loader = build_loader(
        frame=test_df,
        vocabs=vocabs,
        embeddings=embeddings,
        batch_size=1,
        shuffle=False,
        max_queries_per_task=32,
        task_cols=ckpt.get("task_cols", ["seq_id_a", "serumPassCat"]),
        align_ha_embeddings=False,
        include_na_embeddings=False,
    )

    metrics, pred = evaluate_model(model, loader, device)
    pred = pred.copy()
    pred["model"] = model_name
    pred["abs_pred"] = pred["mean"].abs()
    pred["checkpoint_epoch"] = ckpt.get("epoch")
    pred["checkpoint_path"] = str(ckpt_path)

    values = pred["mean"].to_numpy(dtype=float)
    summary = {
        "model": model_name,
        "checkpoint_epoch": ckpt.get("epoch"),
        "rows": len(pred),
        "mean_pred": float(np.mean(values)),
        "median_pred": float(np.median(values)),
        "std_pred": float(np.std(values)),
        "mae_to_zero": float(np.mean(np.abs(values))),
        "mse_to_zero": float(np.mean(values ** 2)),
        "min_pred": float(np.min(values)),
        "p05_pred": float(np.quantile(values, 0.05)),
        "p25_pred": float(np.quantile(values, 0.25)),
        "p75_pred": float(np.quantile(values, 0.75)),
        "p95_pred": float(np.quantile(values, 0.95)),
        "max_pred": float(np.max(values)),
        "pct_abs_le_0.10": float(np.mean(np.abs(values) <= 0.10)),
        "pct_abs_le_0.25": float(np.mean(np.abs(values) <= 0.25)),
        "pct_abs_le_0.50": float(np.mean(np.abs(values) <= 0.50)),
        "mean_log_var": float(pred["log_var"].mean()),
        "mean_self_score": float(pred["self_score"].mean()),
        "mean_query_score": float(pred["query_score"].mean()),
    }
    summary.update({f"metric_{key}": value for key, value in metrics.items()})
    return summary, pred


In [5]:
summary_rows = []
prediction_frames = []

for model_name, ckpt_path in MODELS.items():
    summary, pred = run_checkpoint(model_name, ckpt_path)
    summary_rows.append(summary)
    prediction_frames.append(pred)

    pred_path = OUT_DIR / f"{model_name}_self_pair_H1N1_predictions.csv"
    pred.to_csv(pred_path, index=False)
    print(f"saved {pred_path}")

summary_df = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / "self_pair_H1N1_summary.csv"
# summary_df.to_csv(summary_path, index=False)

all_predictions = pd.concat(prediction_frames, ignore_index=True)
all_predictions_path = OUT_DIR / "self_pair_H1N1_predictions_all_models.csv"
# all_predictions.to_csv(all_predictions_path, index=False)

print(f"saved {summary_path}")
print(f"saved {all_predictions_path}")

summary_df.sort_values("mae_to_zero")


saved /home/chenyh/workspace/fluProfiler/results/H1H3_HA1/self_pair_same_context_H1N1_inference/Minus128_self_pair_H1N1_predictions.csv
saved /home/chenyh/workspace/fluProfiler/results/H1H3_HA1/self_pair_same_context_H1N1_inference/Minus16_self_pair_H1N1_predictions.csv
saved /home/chenyh/workspace/fluProfiler/results/H1H3_HA1/self_pair_same_context_H1N1_inference/Minus8_self_pair_H1N1_predictions.csv
saved /home/chenyh/workspace/fluProfiler/results/H1H3_HA1/self_pair_same_context_H1N1_inference/self_pair_H1N1_summary.csv
saved /home/chenyh/workspace/fluProfiler/results/H1H3_HA1/self_pair_same_context_H1N1_inference/self_pair_H1N1_predictions_all_models.csv


,model,checkpoint_epoch,rows,mean_pred,median_pred,std_pred,mae_to_zero,mse_to_zero,min_pred,p05_pred,...,metric_per_serum_mae_mean,metric_per_serum_mae_median,metric_within_serum_pearson_mean,metric_within_serum_spearman_mean,metric_serum_bias_mean,metric_serum_bias_abs_mean,metric_nll,metric_coverage_80,metric_coverage_95,metric_loss
2,Minus8,200,500,3.237724e-07,2.980232e-07,3.873723e-07,3.237724e-07,2.548859e-13,0.0,0.000000e+00,...,3.510183e-07,2.384186e-07,0.0,0.0,3.510183e-07,3.510183e-07,-0.426872,1.0,1.0,-0.519511
1,Minus16,100,500,5.558878e-07,5.364418e-07,2.025388e-07,5.558878e-07,3.500332e-13,0.0,2.235174e-07,...,4.715339e-07,4.768372e-07,0.0,0.0,4.715339e-07,4.715339e-07,-0.216437,1.0,1.0,-0.307344
0,Minus128,100,500,1.228690e-06,1.192093e-06,2.827108e-07,1.228690e-06,1.589605e-12,0.0,5.960464e-07,...,8.638149e-07,1.072884e-06,0.0,0.0,8.638149e-07,8.638149e-07,-0.449459,1.0,1.0,-0.610128


In [6]:
# Optional quick inspection: rows with the largest absolute self-pair violation per model.
(
    all_predictions
    .sort_values(["model", "abs_pred"], ascending=[True, False])
    .groupby("model", as_index=False)
    .head(10)
    [["model", "serumName", "serumPassCat", "label", "mean", "abs_pred", "self_score", "query_score", "log_var"]]
)


,model,serumName,serumPassCat,label,mean,abs_pred,self_score,query_score,log_var
166,Minus128,A/ISRAEL/Q-504/2015,<CELL>,0.0,0.000002,0.000002,1.052343,1.052341,-0.635749
167,Minus128,A/ISRAEL/Q-504/2015,<CELL>,0.0,0.000002,0.000002,1.052343,1.052341,-0.635749
168,Minus128,A/ISRAEL/Q-504/2015,<CELL>,0.0,0.000002,0.000002,1.052343,1.052341,-0.635749
169,Minus128,A/ISRAEL/Q-504/2015,<CELL>,0.0,0.000002,0.000002,1.052343,1.052341,-0.635749
170,Minus128,A/ISRAEL/Q-504/2015,<CELL>,0.0,0.000002,0.000002,1.052343,1.052341,-0.635749
171,Minus128,A/ISRAEL/Q-504/2015,<CELL>,0.0,0.000002,0.000002,1.052343,1.052341,-0.635749
172,Minus128,A/ISRAEL/Q-504/2015,<CELL>,0.0,0.000002,0.000002,1.052343,1.052341,-0.635749
173,Minus128,A/ISRAEL/Q-504/2015,<CELL>,0.0,0.000002,0.000002,1.052343,1.052341,-0.635749
174,Minus128,A/ISRAEL/Q-504/2015,<CELL>,0.0,0.000002,0.000002,1.052343,1.052341,-0.635749
175,Minus128,A/ISRAEL/Q-504/2015,<CELL>,0.0,0.000002,0.000002,1.052343,1.052341,-0.635749
